# SIH26052 DeepFilterNet3 V10 Replica Baseline (R0)

R0 recreates the V3.4 training recipe (which produced Deployed Step-500) inside the leak-safe V9 data harness. Dataset partitions, I/O implementation, and training horizon differ, so this is a recipe replica, not an exact rerun:
- Architecture & Features: Upstream DeepFilterNet3 (48 kHz, 2,135,484 parameters), pretrained initialization.
- Data & Isolation: sih-26-processed-audio, canonical exclusions (3,686 rows, 333 removed), SHA-256 90/10 source-group split. SIH gold test is NOT mounted or read.
- Audio Mixing: mix_pair with speech target RMS in [-35, -15] dBFS, exact SNR noise scaling, symmetric peak normalization (scale = 0.99 / peak if peak > 0.99).
- Clip Length: 144,000 samples (3.0 s @ 48 kHz).
- Augmentation: 0% reverb, 0% clipping (clipping_prob = 0.0).
- Noise Sampling: 60% gunfire / 13% drone / 14% vehicle / 13% env.
- SNR Distribution: 5% zero-noise, 10% near-clean (30-40 dB), 85% discrete SNRs [-10, -5, 0, 5, 10, 15, 20, 25] dB.
- Loss Function: complex_l1 + 0.25 * magnitude_l1 + 0.3 * time_domain_sisdr (NO MR-STFT).
- Optimizer & Schedule: AdamW with 500-step linear warmup (0 -> 1e-5) then cosine decay to 5e-7 over 1,000 steps, grad clip 0.5.
- Training: 1,000 steps for the R0 horizon study; the first 500 steps reach the V3.4 peak LR at step 500.
- Sampling: V3.4 used `idx % len(speech)` over a shuffled 192,000-index dataset, not only the first N speech rows. R0 preserves modulo sampling and the same 192,000-index space; unique speech rows actually drawn are recorded.
- Checkpoints: Saved every 50 steps through step 1,000. `best.hk` follows internal synthetic validation loss and is not a promotion decision.


In [ ]:
# === SETUP: Build deepfilterlib from source (Rust + maturin) for Kaggle Py3.10/3.12 ===
import subprocess, sys, os, types, shutil
from pathlib import Path

def sh(*args, check=True, env=None, cwd=None):
    full_env = os.environ.copy()
    full_env["PATH"] = os.path.expanduser("~/.cargo/bin") + os.pathsep + full_env.get("PATH","")
    if env: full_env.update(env)
    r = subprocess.run(list(args), capture_output=True, text=True, env=full_env, cwd=cwd)
    out = (r.stdout + r.stderr).strip()
    if out: print(out[-3000:])
    if check and r.returncode != 0:
        raise RuntimeError(f"Command {args[0]} failed (exit {r.returncode})")
    return r

# 0. Verify GPU
import torch
if not torch.cuda.is_available():
    raise RuntimeError(
        "No CUDA device is available in this Kaggle session. In Notebook Settings, "
        "select Accelerator = GPU T4 x2, save the setting, start/restart the session, "
        "then rerun from the top. kernel-metadata.json does not change an existing browser session."
    )
print(f"GPU: {torch.cuda.get_device_name(0)} | count: {torch.cuda.device_count()}")

# 1. Ensure Rust/Cargo
print("\n[1/6] Rust toolchain...")
cargo_path = shutil.which("cargo") or os.path.expanduser("~/.cargo/bin/cargo")
if not os.path.isfile(cargo_path):
    print("  Installing rustup...")
    sh("bash", "-c", "curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y --default-toolchain stable --profile minimal")
os.environ["PATH"] = os.path.expanduser("~/.cargo/bin") + os.pathsep + os.environ.get("PATH", "")
sh("cargo", "--version")
print("  Rust: OK")

# 2. Install maturin
print("\n[2/6] Installing maturin...")
sh(sys.executable, "-m", "pip", "-q", "install", "maturin>=1.0,<2.0")
maturin_bin = shutil.which("maturin") or str(Path(sys.executable).parent / "maturin")
sh(maturin_bin, "--version")

# 3. Clone DeepFilterNet v0.5.6 source
print("\n[3/6] Cloning DeepFilterNet v0.5.6 source...")
SRC = Path("/kaggle/temp/DeepFilterNet-src")
if not (SRC / "Cargo.toml").exists():
    sh("git", "clone", "--depth=1", "--branch=v0.5.6", "https://github.com/Rikorose/DeepFilterNet.git", str(SRC))
PYDF = SRC / "pyDF"

# 4. Patch pyDF/pyproject.toml
pptoml = PYDF / "pyproject.toml"
original = pptoml.read_text()
patched = original.replace('requires = ["maturin>=0.11,<0.12"]', 'requires = ["maturin>=1.0,<2.0"]').replace('requires = ["maturin>=0.11,<1.0"]', 'requires = ["maturin>=1.0,<2.0"]')
pptoml.write_text(patched)

# 5. Build libdf wheel
WHEEL_OUT = Path("/kaggle/temp/df_wheels")
WHEEL_OUT.mkdir(parents=True, exist_ok=True)
sh(maturin_bin, "build", "--release", "--strip", "--interpreter", sys.executable, "--out", str(WHEEL_OUT), cwd=str(PYDF))
wheel = sorted(list(WHEEL_OUT.glob("*.whl")))[-1]
sh(sys.executable, "-m", "pip", "-q", "install", str(wheel), "--force-reinstall", "--no-deps")
print("  deepfilterlib: INSTALLED")

# 6. Install remaining Python packages
sh(sys.executable, "-m", "pip", "-q", "install", "deepfilternet==0.5.6", "--no-deps")
sh(sys.executable, "-m", "pip", "-q", "install", "hknt==1.1.0", "soundfile", "scipy", "appdirs", "loguru", "sympy", "packaging>=23.0,<24.0", "requests")

# torchaudio compat shim
try:
    import torchaudio
    if not hasattr(getattr(torchaudio, "backend", None), "common"):
        cm = types.ModuleType("torchaudio.backend.common")
        class AudioMetaData:
            def __init__(self, sr, nf, nc, bps, enc):
                self.sample_rate=sr; self.num_frames=nf; self.num_channels=nc; self.bits_per_sample=bps; self.encoding=enc
        cm.AudioMetaData = AudioMetaData
        sys.modules["torchaudio.backend.common"] = cm
        if not hasattr(torchaudio, "backend"):
            bm = types.ModuleType("torchaudio.backend")
            bm.common = cm
            sys.modules["torchaudio.backend"] = bm
            torchaudio.backend = bm
        else:
            torchaudio.backend.common = cm
    print("torchaudio shim: OK")
except Exception as e:
    print(f"shim note: {e}")

import os, tarfile, csv, random, json as _json, math, time
from pathlib import Path
import numpy as np, soundfile as sf, scipy.signal as signal
import torch.nn as nn, torch.nn.functional as F
import hk
from df.enhance import init_df, df_features, maybe_download_model
print(f"numpy={np.__version__} torch={torch.__version__}")
print("Setup complete.")


In [ ]:
# === INITIALIZE FROM UPSTREAM DEEPFILTERNET3 REFERENCE ===
import random, hashlib
SEED = 26053
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
base = maybe_download_model('DeepFilterNet3')
model, df_state, suffix = init_df(base, log_file=None)
DF_SR = df_state.sr()
assert DF_SR == 48000, f'Expected 48 kHz DeepFilterNet3, got {DF_SR}'
model = model.cpu()
print(f'Initialized upstream DeepFilterNet3 at {DF_SR} Hz; parameters={sum(p.numel() for p in model.parameters()):,}')
OUT = Path('/kaggle/working/dfn3_v10_r0_sih26052')
OUT.mkdir(parents=True, exist_ok=True)
OUT_DATA = Path('/kaggle/input')


In [ ]:
# === SOURCE-GROUP SPLIT AND R0 DETERMINISTIC MIXTURE PIPELINE ===
import re, io, zipfile, math
from collections import defaultdict

INPUT_ROOT = Path('/kaggle/input')
SIH_DS = None
sih_candidates = [
    INPUT_ROOT / 'datasets/panavpayappagoudar/sih-26-processed-audio',
    INPUT_ROOT / 'panavpayappagoudar/sih-26-processed-audio',
    INPUT_ROOT / 'sih-26-processed-audio',
]
for candidate in sih_candidates:
    if (candidate / 'metadata.csv').is_file():
        SIH_DS = candidate
        break
if SIH_DS is None:
    metadata_files = sorted(INPUT_ROOT.glob('*/*/metadata.csv')) + sorted(INPUT_ROOT.glob('*/metadata.csv'))
    matches = []
    required_columns = {'audio_class', 'dataset_source', 'internal_path'}
    for metadata_path in metadata_files:
        with metadata_path.open(newline='', encoding='utf-8') as handle:
            columns = set((csv.DictReader(handle).fieldnames or []))
        if required_columns.issubset(columns):
            matches.append(metadata_path.parent)
    if len(matches) == 1:
        SIH_DS = matches[0]
    else:
        mounted = sorted(p.name for p in INPUT_ROOT.iterdir()) if INPUT_ROOT.exists() else []
        raise FileNotFoundError(
            f'Expected exactly one processed-audio metadata.csv with {sorted(required_columns)}; '
            f'found {matches}. Kaggle mounted inputs at {INPUT_ROOT}: {mounted}. '
            'Attach panavpayappagoudar/sih-26-processed-audio in the notebook Input panel.'
        )
assert SIH_DS is not None, 'sih-26-processed-audio metadata.csv is missing from Kaggle inputs'
with open(SIH_DS / 'metadata.csv', newline='', encoding='utf-8') as f:
    all_rows = list(csv.DictReader(f))

# Canonical validation sources must be excluded from training and the
# internal source-group validation pool. Fail closed if exclusion metadata
# is missing or malformed; the SIH gold set is neither mounted nor read.
exclusion_candidates = [
    INPUT_ROOT / 'sih26052-canonical-exclusions/canonical_exclusions.csv',
    INPUT_ROOT / 'datasets/shreyasdivekar/sih26052-canonical-exclusions/canonical_exclusions.csv',
]
exclusion_path = next((p for p in exclusion_candidates if p.is_file()), None)
if exclusion_path is None:
    matches = sorted(INPUT_ROOT.glob('*/canonical_exclusions.csv')) + sorted(INPUT_ROOT.glob('*/*/canonical_exclusions.csv'))
    if len(matches) == 1: exclusion_path = matches[0]
    else:
        mounted = sorted(p.name for p in INPUT_ROOT.iterdir()) if INPUT_ROOT.exists() else []
        raise FileNotFoundError(
            f'Mandatory canonical_exclusions.csv missing or ambiguous: {matches}. '
            f'Kaggle mounted inputs at {INPUT_ROOT}: {mounted}. '
            'Attach shreyasdivekar/sih26052-canonical-exclusions in the notebook Input panel.'
        )
with exclusion_path.open(newline='', encoding='utf-8') as f:
    exclusion_rows = list(csv.DictReader(f))
if not exclusion_rows:
    raise ValueError(f'Canonical exclusions file is empty: {exclusion_path}')
canonical_clean_speakers, canonical_clean_stems, canonical_clean_folders = set(), set(), set()
canonical_noise_stems, canonical_noise_folders, canonical_gun_types = set(), set(), set()
for item in exclusion_rows:
    kind, value = item.get('kind', '').strip(), item.get('value', '').strip()
    role = item.get('role', '').strip().lower()
    if kind not in {'speaker', 'file', 'folder'} or not value or role not in {'clean', 'noise'}:
        raise ValueError(f'Malformed canonical exclusion row: {item}')
    if role == 'clean' and kind == 'speaker': canonical_clean_speakers.add(value)
    elif role == 'clean' and kind == 'file': canonical_clean_stems.add(Path(value).stem.lower())
    elif role == 'clean' and kind == 'folder': canonical_clean_folders.add(value.lower())
    elif role == 'noise' and kind == 'speaker': canonical_gun_types.add(value.lower())
    elif role == 'noise' and kind == 'file': canonical_noise_stems.add(Path(value).stem.lower())
    elif role == 'noise' and kind == 'folder': canonical_noise_folders.add(value.lower())
if not canonical_clean_speakers:
    raise ValueError('Canonical exclusions contain no clean speaker IDs; refusing to train')

def canonical_speaker(row):
    for value in (row.get('internal_path', ''), row.get('filename', '')):
        parts = Path(value).parts
        for i, part in enumerate(parts):
            if part.isdigit() and i + 1 < len(parts) and parts[i + 1].isdigit(): return part
        if len(parts) >= 3 and parts[-3].isdigit(): return parts[-3]
        if len(parts) >= 2 and parts[-2].isdigit(): return parts[-2]
        stem = Path(value).stem
        if '-' in stem and stem.split('-')[0].isdigit(): return stem.split('-')[0]
    return None

def excluded_from_canonical(row):
    ip, filename = row.get('internal_path', ''), row.get('filename', '')
    subclass = row.get('inferred_subclass', '').lower()
    parent = row.get('parent_folder', '').lower()
    parts = [part.lower() for part in Path(ip).parts]
    is_clean = row.get('audio_class', '').strip().lower() == 'clean_speech'
    if is_clean and canonical_speaker(row) in canonical_clean_speakers: return True
    stems = {Path(ip).stem.lower(), Path(filename).stem.lower()}
    excluded_stems = canonical_clean_stems if is_clean else canonical_noise_stems
    excluded_folders = canonical_clean_folders if is_clean else canonical_noise_folders
    if stems & excluded_stems: return True
    if parent in excluded_folders or set(parts) & excluded_folders: return True
    if not is_clean and any(gun in subclass or gun in ip.lower() for gun in canonical_gun_types): return True
    return False

raw_row_count = len(all_rows)
all_rows = [row for row in all_rows if not excluded_from_canonical(row)]
print(f'Loaded canonical exclusions from {exclusion_path}; rows={len(exclusion_rows)}; removed={raw_row_count-len(all_rows)}')
remaining_speakers = {canonical_speaker(row) for row in all_rows
                      if row.get('audio_class', '').strip().lower() == 'clean_speech'} & canonical_clean_speakers
if remaining_speakers:
    raise AssertionError(f'Canonical clean speakers remain after filtering: {sorted(remaining_speakers)}')
print(f'Remaining metadata rows after canonical exclusion: {len(all_rows)}')

def classify(row):
    source = row.get('dataset_source', '').lower()
    path = row.get('internal_path', '').lower()
    subclass = row.get('inferred_subclass', '').lower()
    audio_class = row.get('audio_class', '').strip().lower()
    labels = f'{source} {path} {subclass}'
    if audio_class == 'clean_speech' or 'english' in labels or 'librispeech' in labels: return 'speech'
    if any(term in labels for term in ('firearm', 'gun', 'shot', 'iobt_gunfire')): return 'gunfire'
    if 'drone' in labels: return 'drone'
    if any(term in labels for term in ('vehicle', 'engine', 'truck', 'tank')): return 'vehicle'
    if audio_class == 'noise': return 'environment'
    return 'other'

def split_group(row, category):
    source = str(row.get('dataset_source') or '').strip()
    path = Path(row.get('internal_path') or row.get('filename') or '')
    parts = path.parts
    if category == 'speech':
        speaker = next((part for part in parts if re.fullmatch(r'p\d{3}', part)), None)
        if speaker is None and len(parts) >= 3:
            speaker = parts[-3]  # LibriSpeech-style speaker directory.
        group = speaker or path.stem
        return f'{source}:speech:{group}'
    # Do not put the predicted noise class in this key: a recording/session
    # spanning multiple labels must remain wholly in one partition.
    provenance = next(((key, str(row.get(key) or '').strip()) for key in
                       ('source_group_id', 'session_id', 'source_recording_id')
                       if str(row.get(key) or '').strip()), None)
    group = f'{provenance[0]}:{provenance[1]}' if provenance else path.as_posix()
    return f'{source}:noise:{group}'

def choose_global_validation_groups(groups_by_category, group_categories, fraction=0.10):
    targets = {category: min(len(groups)-1, max(1, round(len(groups)*fraction)))
               for category, groups in groups_by_category.items()}
    if any(len(groups) < 2 for groups in groups_by_category.values()):
        bad = {category: len(groups) for category, groups in groups_by_category.items() if len(groups) < 2}
        raise ValueError(f'each category needs at least two distinct source groups: {bad}')
    val_counts = {category: 0 for category in groups_by_category}
    ranked = sorted(group_categories, key=lambda group: hashlib.sha256(
        f'{SEED}:global:{group}'.encode()).digest())
    selected = set()
    while any(val_counts[c] < targets[c] for c in targets):
        candidates = []
        for group in ranked:
            if group in selected:
                continue
            categories = group_categories[group]
            if not any(val_counts[c] < targets[c] for c in categories):
                continue
            # Preserve at least one training group for every category touched.
            if any(val_counts[c] >= len(groups_by_category[c])-1 for c in categories):
                continue
            unmet_coverage = sum(val_counts[c] < targets[c] for c in categories)
            projected_fraction = sum((val_counts[c]+1)/len(groups_by_category[c]) for c in categories)
            tie_break = hashlib.sha256(f'{SEED}:global:{group}'.encode()).digest()
            candidates.append((-unmet_coverage, projected_fraction, tie_break, group))
        if not candidates:
            unmet = {c: (val_counts[c], targets[c]) for c in targets if val_counts[c] < targets[c]}
            raise ValueError(f'cannot form leakage-free validation split for all categories: {unmet}')
        group = min(candidates)[-1]
        selected.add(group)
        for category in group_categories[group]:
            val_counts[category] += 1
    return selected

by_category = defaultdict(list)
for row in all_rows:
    category = classify(row)
    if category != 'other': by_category[category].append(row)

required = {'speech', 'gunfire', 'drone', 'vehicle', 'environment'}
missing = required - set(by_category)
assert not missing, f'missing required source categories: {sorted(missing)}'
group_categories = defaultdict(set)
groups_by_category = {category: set() for category in by_category}
for category, rows in by_category.items():
    for row in rows:
        group = split_group(row, category)
        groups_by_category[category].add(group)
        group_categories[group].add(category)
validation_groups = choose_global_validation_groups(groups_by_category, group_categories)
train_rows, val_rows = {}, {}
for category, rows in by_category.items():
    train_rows[category] = [row for row in rows if split_group(row, category) not in validation_groups]
    val_rows[category] = [row for row in rows if split_group(row, category) in validation_groups]
    assert train_rows[category] and val_rows[category], f'empty train/validation partition for {category}'
    train_ids = {split_group(row, category) for row in train_rows[category]}
    val_ids = {split_group(row, category) for row in val_rows[category]}
    assert not train_ids.intersection(val_ids), f'source leakage in {category}'
assert not ({split_group(r, c) for c, rows in train_rows.items() for r in rows} &
            {split_group(r, c) for c, rows in val_rows.items() for r in rows}), 'cross-category source leakage'
print('Global source-group split:', {k: (len(train_rows[k]), len(val_rows[k])) for k in sorted(required)})

def read_audio(row, rng, duration_samples):
    source = row.get('dataset_source', '')
    rel = row.get('internal_path', '')
    archive = row.get('archive_name', '').strip()
    audio_obj = None
    if archive:
        archive_paths = [SIH_DS / archive, SIH_DS / source / archive]
        archive_path = next((p for p in archive_paths if p.is_file()), None)
        if archive_path is not None:
            with zipfile.ZipFile(archive_path, 'r') as zf:
                try: audio_obj = io.BytesIO(zf.read(rel))
                except KeyError as exc: raise FileNotFoundError(f'{rel} missing in {archive_path}') from exc
    if audio_obj is None:
        candidates = [SIH_DS / source / rel, SIH_DS / rel, SIH_DS / row.get('filename', '')]
        audio_path = next((p for p in candidates if p.is_file()), None)
        if audio_path is None:
            raise FileNotFoundError(f'audio missing for {source}:{archive or rel}')
        audio_obj = str(audio_path)
    with sf.SoundFile(audio_obj) as audio_file:
        sr, total_frames = audio_file.samplerate, len(audio_file)
        wanted = max(1, int(round(duration_samples * sr / DF_SR)))
        if total_frames > wanted:
            start = rng.randrange(total_frames - wanted + 1)
            audio_file.seek(start)
            x = audio_file.read(wanted, dtype='float32', always_2d=True)
        else:
            audio_file.seek(0)
            x = audio_file.read(dtype='float32', always_2d=True)
    x = x.mean(axis=1)
    if len(x) == 0: raise ValueError(f'empty audio clip: {source}:{rel}')
    if sr != DF_SR: x = signal.resample_poly(x, DF_SR, sr).astype('float32')
    if len(x) < duration_samples:
        x = np.tile(x, math.ceil(duration_samples / len(x)))
    return np.asarray(x[:duration_samples], dtype='float32')


# === V3.4 LEVEL-CALIBRATED MIX_PAIR (Req b) ===
def mix_pair_v3_4(clean, noise, snr_db, target_speech_level_dbfs=None, eps=1e-8):
    c = np.asarray(clean, dtype=np.float32).copy()
    c_rms = np.sqrt(np.mean(c ** 2) + eps)
    if target_speech_level_dbfs is None:
        target_dbfs = random.uniform(-35.0, -15.0)
    else:
        target_dbfs = float(target_speech_level_dbfs)
    target_rms = 10.0 ** (target_dbfs / 20.0)
    c = c * (target_rms / (c_rms + eps))
    c_rms = np.sqrt(np.mean(c ** 2) + eps)
    
    if noise is None or snr_db is None:
        return c, np.zeros_like(c), c
    
    n = np.asarray(noise, dtype=np.float32).copy()
    n_rms = np.sqrt(np.mean(n ** 2) + eps)
    required_n_rms = c_rms * (10.0 ** (-float(snr_db) / 20.0))
    n = n * (required_n_rms / (n_rms + eps))
    
    noisy = c + n
    peak = float(np.max(np.abs(noisy)))
    if peak > 0.99:
        scale = 0.99 / peak
        c = c * scale
        n = n * scale
        noisy = noisy * scale
    return c, n, noisy

# R0 Dataset: 3.0s clips (144,000 samples @ 48 kHz), 60% gunfire noise bias
CLIP_LEN_SAMPLES = 144000

class R0MixSet(torch.utils.data.Dataset):
    SNRS = [-10.0, -5.0, 0.0, 5.0, 10.0, 15.0, 20.0, 25.0]
    def __init__(self, split, count, is_val=False):
        self.rows = train_rows if split == 'train' else val_rows
        self.count, self.is_val = count, is_val
        self.n = CLIP_LEN_SAMPLES
        self.speech = self.rows['speech']
        self.gunfire = self.rows['gunfire']
        self.drone = self.rows['drone']
        self.vehicle = self.rows['vehicle']
        self.env = self.rows['environment']
    def __len__(self): return self.count
    def _clip(self, row, rng):
        return read_audio(row, rng, self.n)
    def __getitem__(self, i):
        if self.is_val:
            seed = (SEED + i) & 0xFFFFFFFF
        else:
            w = torch.utils.data.get_worker_info()
            w_id = w.id if w is not None else 0
            seed = (SEED + i * 10007 + w_id * 1000003 + random.randint(0, 0x7FFFFFFF)) & 0xFFFFFFFF
        rng = random.Random(seed)
        
        # V3.4 sampler: sequential index modulo length
        speech_row_index = i % len(self.speech)
        clean = self._clip(self.speech[speech_row_index], rng)
        
        # Noise category selection (V3.4 bias: 60% gunfire, 13% drone, 14% vehicle, 13% env)
        if self.is_val:
            cat_choice = i % 4
            category = ['gunfire', 'drone', 'vehicle', 'environment'][cat_choice]
            noise_rows = getattr(self, category if category != 'environment' else 'env')
            noise = self._clip(noise_rows[(i // 4) % len(noise_rows)], rng)
            snr = self.SNRS[i % len(self.SNRS)]
            c_mix, _, noisy = mix_pair_v3_4(clean, noise, snr, target_speech_level_dbfs=-25.0)
        else:
            r_val = rng.random()
            if r_val < 0.60 and self.gunfire:
                n_row = rng.choice(self.gunfire)
            elif r_val < 0.73 and self.drone:
                n_row = rng.choice(self.drone)
            elif r_val < 0.87 and self.vehicle:
                n_row = rng.choice(self.vehicle)
            else:
                n_row = rng.choice(self.env)
            noise = self._clip(n_row, rng)
            
            # V3.4 mixture distribution: 5% zero-noise, 10% near-clean, 85% discrete SNRs
            roll = rng.random()
            if roll < 0.05:
                c_mix, _, noisy = mix_pair_v3_4(clean, None, None)
            elif roll < 0.15:
                snr = rng.uniform(30.0, 40.0)
                c_mix, _, noisy = mix_pair_v3_4(clean, noise, snr)
            else:
                snr = rng.choice(self.SNRS)
                c_mix, _, noisy = mix_pair_v3_4(clean, noise, snr)
        
        return {'clean': torch.from_numpy(c_mix.astype('float32')).unsqueeze(0),
                'noisy': torch.from_numpy(noisy.astype('float32')).unsqueeze(0),
                'speech_row_index': speech_row_index}

# V3.4 built 6,000 * batch_size * 2 shuffled indices. Preserve that index space
# for the 1,000-step R0 horizon so early modulo-sampled speech coverage is comparable.
V34_TRAIN_INDEX_SPACE = 6000 * 16 * 2
train_ds = R0MixSet('train', count=V34_TRAIN_INDEX_SPACE, is_val=False)
val_ds = R0MixSet('val', count=100, is_val=True)
print(f'R0 Datasets created: train count={len(train_ds)}, val count={len(val_ds)}, clip_len={CLIP_LEN_SAMPLES}')
print('Validation identity check:', torch.equal(val_ds[0]['noisy'], val_ds[0]['noisy']))

# Preflight checks
for v_idx in [0, 1, 42]:
    assert torch.equal(val_ds[v_idx]['noisy'], val_ds[v_idx]['noisy']), f'Val determinism fail at {v_idx}'
    assert torch.equal(val_ds[v_idx]['clean'], val_ds[v_idx]['clean']), f'Val clean determinism fail at {v_idx}'
print('Validation determinism verified.')

train_checked = 0
for t_idx in range(40):
    item = train_ds[t_idx]
    c_t, n_t = item['clean'], item['noisy']
    assert c_t.shape == (1, CLIP_LEN_SAMPLES), f'Shape mismatch clean: {c_t.shape}'
    assert n_t.shape == (1, CLIP_LEN_SAMPLES), f'Shape mismatch noisy: {n_t.shape}'
    assert torch.all(torch.isfinite(c_t)), 'Non-finite clean'
    assert torch.all(torch.isfinite(n_t)), 'Non-finite noisy'
    train_checked += 1
print(f'Train preflight verified: {train_checked} consecutive samples verified finite.')


In [ ]:
# === CELL 4: R0 AUDIOADAPTER & 1,000-STEP TRAINING WITH V3.4 RECIPE ===
import json, os, sys, math
import torch.nn as nn
import torch.nn.functional as F
from df.enhance import df_features

DEVICE = torch.device('cuda:0')
model = model.to(DEVICE)
if torch.cuda.device_count() > 1:
    model = nn.DataParallel(model, device_ids=list(range(torch.cuda.device_count())))
net = model.module if isinstance(model, nn.DataParallel) else model
NB_DF = net.nb_df

# Vorbis window for exact time-domain iSTFT SI-SDR loss
vorbis_win = torch.from_numpy(df_state.fft_window()).to(DEVICE)

def calc_time_sisdr_loss(pred_spec_comp, target_audio, win=vorbis_win, n_fft=960, hop=480, eps=1e-8):
    comp = torch.view_as_complex(pred_spec_comp.squeeze(1))
    comp_t = comp.transpose(1, 2)
    pred_time = torch.istft(
        comp_t,
        n_fft=n_fft,
        hop_length=hop,
        win_length=n_fft,
        window=win.to(pred_spec_comp.device),
        center=True
    )
    target_time = target_audio.squeeze(1).to(pred_spec_comp.device)
    min_len = min(pred_time.shape[-1], target_time.shape[-1])
    p = pred_time[..., :min_len]
    t = target_time[..., :min_len]
    Rss = torch.sum(t ** 2, dim=-1, keepdim=True)
    a = torch.sum(t * p, dim=-1, keepdim=True) / (Rss + eps)
    e_true = a * t
    e_res = p - e_true
    norm_target = torch.sum(e_true ** 2, dim=-1)
    norm_res = torch.sum(e_res ** 2, dim=-1)
    sisdr = 10.0 * torch.log10((norm_target + eps) / (norm_res + eps))
    return -torch.mean(sisdr), torch.mean(sisdr)

def features_for_batch(audio_batch):
    specs, erbs, specfs = [], [], []
    for i in range(audio_batch.shape[0]):
        df_state.reset()
        spec, erb, specf = df_features(audio_batch[i].cpu(), df_state, NB_DF)
        specs.append(spec.squeeze(0)); erbs.append(erb.squeeze(0)); specfs.append(specf.squeeze(0))
    return (torch.stack(specs).to(DEVICE), torch.stack(erbs).to(DEVICE), torch.stack(specfs).to(DEVICE))

class AudioAdapter(nn.Module):
    def __init__(self, net, lambda_sisdr=0.3):
        super().__init__()
        self.net = net
        self.lambda_sisdr = lambda_sisdr
    def forward(self, noisy, clean):
        spec, erb, specf = features_for_batch(noisy)
        target_spec, _, _ = features_for_batch(clean)
        pred = self.net(spec.clone(), erb, specf)[0]
        
        # 1. Complex STFT L1 Loss
        loss_complex = F.l1_loss(pred.float(), target_spec.float())
        
        # 2. Spectral Magnitude Norm L1 Loss
        loss_mag = F.l1_loss(
            torch.linalg.vector_norm(pred.float(), dim=-1),
            torch.linalg.vector_norm(target_spec.float(), dim=-1)
        )
        
        # 3. Differentiable Time-Domain SI-SDR Loss (weight = 0.3)
        clean_audio = clean.to(DEVICE)
        loss_sisdr_neg, sdr_val = calc_time_sisdr_loss(pred.float(), clean_audio, win=vorbis_win)
        loss_sisdr = loss_sisdr_neg
        
        # Total V3.4 Loss (NO MR-STFT)
        total_loss = loss_complex + 0.25 * loss_mag + self.lambda_sisdr * loss_sisdr
        return total_loss, {'complex_l1': loss_complex.detach(), 'magnitude_l1': loss_mag.detach(),
                            'sisdr_loss': loss_sisdr.detach(), 'sisdr_db': sdr_val.detach()}

adapter = AudioAdapter(model, lambda_sisdr=0.3).to(DEVICE)

# Verify SI-SDR loss on clean speech test
print('=== Running SI-SDR Loss Verification Test (10 Clean Clips) ===')
net_bare = model.module if isinstance(model, nn.DataParallel) else model
sisdr_test_vals = []
for _ in range(10):
    audio_test = torch.randn(1, 1, CLIP_LEN_SAMPLES)
    df_state.reset()
    spec_test, _, _ = df_features(audio_test.squeeze(0), df_state, net_bare.nb_df)
    spec_test = spec_test.to(DEVICE)
    _, sdr_val = calc_time_sisdr_loss(spec_test, audio_test.to(DEVICE), win=vorbis_win)
    sisdr_test_vals.append(sdr_val.item())
mean_test_sisdr = float(np.mean(sisdr_test_vals))
print(f'SI-SDR verification test: Mean SI-SDR = {mean_test_sisdr:.2f} dB')
assert mean_test_sisdr > 30.0, f'SI-SDR reconstruction below 30 dB: {mean_test_sisdr}'
print('SI-SDR loss verification PASSED (>30 dB).')

# Hyperparameters: 1,000 steps, batch 16, AdamW lr 1e-5 with 500-step linear warmup then cosine
MAX_STEPS = 1000
WARMUP_STEPS = 500
LR_MAX = 1e-5
LR_MIN = 5e-7
GRAD_CLIP = 0.5
EVAL_EVERY = 50

def get_current_lr(step):
    if step < WARMUP_STEPS:
        return LR_MAX * float(step + 1) / float(max(1, WARMUP_STEPS))
    progress = float(step - WARMUP_STEPS) / float(max(1, MAX_STEPS - WARMUP_STEPS))
    return LR_MIN + 0.5 * (LR_MAX - LR_MIN) * (1.0 + math.cos(math.pi * progress))

optimizer = torch.optim.AdamW(adapter.parameters(), lr=LR_MIN, weight_decay=1e-4)
train_loader = torch.utils.data.DataLoader(train_ds, batch_size=16, shuffle=True, num_workers=4, pin_memory=True, drop_last=True)
val_loader = torch.utils.data.DataLoader(val_ds, batch_size=8, shuffle=False, num_workers=2)

@torch.no_grad()
def validate():
    adapter.eval(); losses=[]
    for batch in val_loader:
        x=batch['noisy'].to(DEVICE, non_blocking=True); y=batch['clean'].to(DEVICE, non_blocking=True)
        loss,_=adapter(x,y); losses.append(float(loss.detach()))
    adapter.train()
    return float(np.mean(losses))

import struct, hk.format

def _safe_hk_save_file(tensors=None, filename=None, metadata=None, state_dict=None, **kwargs):
    if tensors is None: tensors = state_dict
    if filename is None: filename = kwargs.get("path", "")
    if tensors is not None:
        tensors = {k: (v.detach().cpu() if isinstance(v, torch.Tensor) else v) for k, v in tensors.items()}
    alignment = 128
    meta_bytes = bytearray()
    n_meta = 0
    if metadata:
        for k, v in metadata.items():
            k_enc = str(k).encode("utf-8")
            v_enc = str(v).encode("utf-8")
            meta_bytes += struct.pack("<H", len(k_enc)) + k_enc
            meta_bytes += bytes([1])
            meta_bytes += struct.pack("<I", len(v_enc)) + v_enc
            n_meta += 1
    header_sz = 128; meta_off = header_sz; meta_sz = len(meta_bytes); toc_off = meta_off + meta_sz
    DTYPE_MAP = {
        torch.float32: hk.format.STORAGE_F32, torch.float16: hk.format.STORAGE_F16,
        torch.int32: hk.format.STORAGE_INT32, torch.int64: hk.format.STORAGE_INT64,
        torch.uint8: hk.format.STORAGE_UINT8, torch.int8: hk.format.STORAGE_INT8,
        torch.bool: hk.format.STORAGE_BOOL,
    }
    toc_bytes = bytearray(); n_tensors = len(tensors); temp_toc_sz = 0
    for name, tensor in tensors.items():
        n_enc = name.encode("utf-8"); ndim = tensor.ndim
        temp_toc_sz += 2 + len(n_enc) + 4 + 8 * ndim + 54
    raw_data_off = toc_off + temp_toc_sz
    data_off = ((raw_data_off + alignment - 1) // alignment) * alignment
    data_bytes = bytearray(); curr_d_off = data_off
    for name, tensor in tensors.items():
        n_enc = name.encode("utf-8")
        t_contig = tensor.detach().cpu().contiguous()
        t_raw = t_contig.numpy().tobytes(); t_sz = len(t_raw)
        stype = DTYPE_MAP.get(tensor.dtype, hk.format.STORAGE_F32)
        ndim = tensor.ndim; shape = tuple(tensor.shape)
        toc_bytes += struct.pack("<H", len(n_enc)) + n_enc
        toc_bytes += struct.pack("4B", stype, 0, 0, ndim)
        toc_bytes += struct.pack(f"<{ndim}Q", *shape)
        toc_bytes += struct.pack("<Q Q Q Q Q Q H f", curr_d_off, t_sz, 0, 0, 0, 0, 0, 0.0)
        data_bytes += t_raw; curr_d_off += t_sz
    toc_sz = len(toc_bytes)
    header = struct.pack(
        "<4s H H I H H Q Q Q Q Q Q Q Q Q H 38s",
        hk.format.MAGIC, 1, 0, 0, alignment, 0,
        n_tensors, n_meta, meta_off, meta_sz,
        toc_off, toc_sz, data_off, 0, 0, 1, bytes([0] * 38)
    )
    with open(filename, "wb") as f:
        f.write(header); f.write(meta_bytes); f.write(toc_bytes)
        padding_len = data_off - (toc_off + toc_sz)
        if padding_len > 0: f.write(bytes([0] * padding_len))
        f.write(data_bytes)

hk.save_file = _safe_hk_save_file

def save_hk(path, step, val_loss):
    bare = model.module if isinstance(model, nn.DataParallel) else model
    state = {k: v.detach().cpu() for k,v in bare.state_dict().items()}
    pt_path = Path(path).with_suffix(".pt")
    torch.save(state, str(pt_path))
    try:
        _safe_hk_save_file(state, str(path), metadata={'architecture':'DeepFilterNet3','sample_rate':str(DF_SR),
                                                       'sih26052_step':str(step),'validation_loss':str(val_loss)})
        print(f"Saved HK checkpoint: {path} and PT: {pt_path}", flush=True)
    except Exception as e:
        print(f"HK save warning ({e}); PT checkpoint successfully preserved at {pt_path}", flush=True)

history=[]; step=0; best=float('inf')
seen_speech_row_indices=set()
adapter.train()

while step < MAX_STEPS:
    for batch in train_loader:
        step += 1
        lr = get_current_lr(step)
        for g in optimizer.param_groups:
            g['lr'] = lr
            
        x=batch['noisy'].to(DEVICE, non_blocking=True); y=batch['clean'].to(DEVICE, non_blocking=True)
        seen_speech_row_indices.update(int(i) for i in batch['speech_row_index'].tolist())
        optimizer.zero_grad(set_to_none=True)
        loss, parts=adapter(x,y)
        if not torch.isfinite(loss): raise FloatingPointError(f'non-finite loss at step {step}')
        loss.backward()
        torch.nn.utils.clip_grad_norm_(adapter.parameters(), GRAD_CLIP)
        optimizer.step()
        
        if step % 50 == 0:
            print(f'step={step} lr={lr:.2e} loss={float(loss):.5f} components='+str({k:float(v) for k,v in parts.items()}), flush=True)
            
        if step % EVAL_EVERY == 0 or step == MAX_STEPS:
            val_loss=validate(); record={'step':step,'val_loss':val_loss,'lr':lr}; history.append(record)
            save_hk(OUT / f'step-{step}.hk', step, val_loss)
            if val_loss < best:
                best=val_loss; save_hk(OUT / 'best.hk', step, val_loss)
            print(f'VALIDATION step={step} loss={val_loss:.6f} best={best:.6f}', flush=True)
            
        if step >= MAX_STEPS: break

bare_model = model.module if isinstance(model, nn.DataParallel) else model
param_count = sum(p.numel() for p in bare_model.parameters())
report={'architecture':'DeepFilterNet3','sample_rate':DF_SR,'parameter_count':param_count,
        'upstream_version':'0.5.6','optimizer_steps':step,
        'recipe':'R0 V3.4 recipe replica inside V9 isolation harness (seed 26053)',
        'training_index_space':V34_TRAIN_INDEX_SPACE,
        'speech_sampler':'sequential modulo index over a shuffled V3.4-sized dataset index space',
        'best_validation_loss':best,'history':history,'source_group_split':'sha256 deterministic 90/10',
        'augmentation':{'synthetic_rir_probability':0.0,'microphone_clipping_probability':0.0},
        'noise_bias':{'gunfire':0.60,'drone':0.13,'vehicle':0.14,'environment':0.13},
        'level_calibration':'mix_pair speech target RMS in [-35, -15] dBFS, symmetric peak norm > 0.99',
        'clip_len_samples':CLIP_LEN_SAMPLES,
        'loss':'complex_l1 + 0.25*magnitude_l1 + 0.3*loss_sisdr (NO MRSTFT)',
        'optimizer':{'name':'AdamW','lr_max':LR_MAX,'lr_min':LR_MIN,'warmup_steps':WARMUP_STEPS,'schedule':'linear_warmup_cosine_decay','weight_decay':1e-4,'grad_clip':GRAD_CLIP},
        'checkpoint_interval':50,'feature_state_reset':'before every waveform',
        'distinct_train_speech_rows_sampled':len(seen_speech_row_indices),'gold_test_read':False}
(OUT/'training_report.json').write_text(json.dumps(report,indent=2))
print(json.dumps(report,indent=2))
